# Task 1 — Measure What Our Judge Can Actually Detect

## Objective

Evaluate whether the LLM judge reliably detects specific errors in English-to-Mandarin translations.

## Method

1. Select translations that previously received high judge scores.
2. Create controlled faulty translations, changing one aspect at a time.
3. Include a punctuation-only control.
4. Evaluate clean and modified translations using the same judge.
5. Calculate score drops by fault type.
6. Compare observed drops with predefined expectations.
7. Report detected, missed, and skipped faults separately.

## Judge Configuration

- Model: Qwen/Qwen3-4B
- Prompt: P2 Anchored
- Original score scale: 1–5
- Temperature: 0.7
- Repeated runs: 3

## Deliverables

- Fault-detection results file
- Short written finding

In [2]:
import torch

In [3]:
import pandas as pd
import numpy as np

eval_df = pd.read_csv(
    "wmt17_en_zh_judge_eval_50.csv"
)

scores_df = pd.read_csv(
    "judge_score_variability_results.csv"
)

# Use temperature 0.7 results
temp_scores = scores_df[
    np.isclose(scores_df["temperature"], 0.7)
]

# Average judge score per translation
mean_scores = (
    temp_scores.groupby("eval_id")["score"]
    .mean()
    .reset_index(name="clean_mean_score")
)

# Join translations with scores
candidates = eval_df.merge(
    mean_scores,
    on="eval_id",
    how="inner"
)

# Select highly rated translations
high_scoring = candidates[
    candidates["clean_mean_score"] >= 4.0
].copy()

print("Total examples:", len(candidates))
print("High-scoring examples:", len(high_scoring))

display(
    high_scoring[
        [
            "eval_id",
            "source_en",
            "candidate_zh",
            "clean_mean_score"
        ]
    ].head(10)
)


Total examples: 50
High-scoring examples: 36


,eval_id,source_en,candidate_zh,clean_mean_score
0,E001,Democratic members of the House Intelligence C...,"众议院情报委员会的民主成员在星期四公布了他们自己的调查结果,他们同意共和党工作队的一些结论。",4.0
2,E003,Gee and Schwartzman are scheduled to discuss a...,Gee和Schwartzman计划讨论和分享Bertram Goodhue 1926年设计以...,4.0
3,E004,"Over the past decade, the State Council has is...","过去十年来,国务院共发出五份消除过度和落后生产能力的通知。",4.0
4,E005,Helen Glover and Heather Stanning inspired to ...,Helen Glover和Heather Stanning在教练的癌症战役中 获得了划划船胜...,4.0
5,E006,12-0 and GB had hardly touched the ball.,12-0和GB几乎没有碰过球。,4.0
7,E008,Even though she learns and works in decentrali...,"尽管她以分散、模块化的方式学习和工作,但她基本上能够完成任务,并取得良好成果。",4.0
9,E010,According to the forecast of Ministry of Trade...,"根据贸易和经济发展部的预测,该国的经济增长将达到7%,而该国的通货膨胀率基本上高于这一水平。",4.0
10,E011,For Ankara the benefits of calling a truce are...,"对安卡拉来说,要求休战的好处是显而易见的。",4.0
11,E012,"Sun Jianbo, the winner of national moral model...","国家道德模范提名奖得主孙建博(Sun Jianbo)和元山林业中心前党书记说,雷凤精神继承了...",4.0
14,E015,The idiot didn't realize they were still on th...,那个白痴不知道他们还在直播,4.0


In [4]:

fault_cases = []

def add_fault(eval_id, fault_type, faulty_translation, explanation):
    row = high_scoring.loc[
        high_scoring["eval_id"] == eval_id
    ]

    assert len(row) == 1, f"Example {eval_id} not found"

    row = row.iloc[0]

    fault_cases.append({
        "eval_id": eval_id,
        "fault_type": fault_type,
        "source_en": row["source_en"],
        "reference_zh": row["reference_zh"],
        "clean_translation": row["candidate_zh"],
        "faulty_translation": faulty_translation,
        "modification": explanation,
        "status": "APPLIED"
    })


# 1. Negation flip
add_fault(
    "E008",
    "negation",
    "尽管她以分散、模块化的方式学习和工作,但她基本上不能够完成任务,并取得良好成果。",
    "Changed 能够 to 不能够, reversing ability."
)

# 2. Number swap
add_fault(
    "E010",
    "entity_number",
    "根据贸易和经济发展部的预测,该国的经济增长将达到9%,而该国的通货膨胀率基本上高于这一水平。",
    "Changed 7% to 9%."
)

# 3. Dropped clause
add_fault(
    "E001",
    "dropped_clause",
    "众议院情报委员会的民主成员在星期四公布了他们自己的调查结果。",
    "Removed the clause about agreeing with Republican conclusions."
)

# 4. Truncation
add_fault(
    "E004",
    "truncation",
    "过去十年来,国务院共发出五份",
    "Removed the ending of the translation."
)

# 5. Untranslated English output
row = high_scoring.loc[
    high_scoring["eval_id"] == "E011"
].iloc[0]

add_fault(
    "E011",
    "untranslated",
    row["source_en"],
    "Returned the original English source instead of Mandarin."
)

# 6. Punctuation-only control
add_fault(
    "E006",
    "punctuation_control",
    "12-0和GB几乎没有碰过球!",
    "Changed the final punctuation mark only."
)

fault_df = pd.DataFrame(fault_cases)

print("Total applied fault cases:", len(fault_df))

display(
    fault_df[
        [
            "eval_id",
            "fault_type",
            "clean_translation",
            "faulty_translation",
            "modification"
        ]
    ]
)

fault_df.to_csv(
    "fault_cases_draft.csv",
    index=False,
    encoding="utf-8-sig"
)


Total applied fault cases: 6


,eval_id,fault_type,clean_translation,faulty_translation,modification
0,E008,negation,"尽管她以分散、模块化的方式学习和工作,但她基本上能够完成任务,并取得良好成果。","尽管她以分散、模块化的方式学习和工作,但她基本上不能够完成任务,并取得良好成果。","Changed 能够 to 不能够, reversing ability."
1,E010,entity_number,"根据贸易和经济发展部的预测,该国的经济增长将达到7%,而该国的通货膨胀率基本上高于这一水平。","根据贸易和经济发展部的预测,该国的经济增长将达到9%,而该国的通货膨胀率基本上高于这一水平。",Changed 7% to 9%.
2,E001,dropped_clause,"众议院情报委员会的民主成员在星期四公布了他们自己的调查结果,他们同意共和党工作队的一些结论。",众议院情报委员会的民主成员在星期四公布了他们自己的调查结果。,Removed the clause about agreeing with Republi...
3,E004,truncation,"过去十年来,国务院共发出五份消除过度和落后生产能力的通知。","过去十年来,国务院共发出五份",Removed the ending of the translation.
4,E011,untranslated,"对安卡拉来说,要求休战的好处是显而易见的。",For Ankara the benefits of calling a truce are...,Returned the original English source instead o...
5,E006,punctuation_control,12-0和GB几乎没有碰过球。,12-0和GB几乎没有碰过球!,Changed the final punctuation mark only.


In [5]:

# Review candidates from our existing dataset
# without creating or modifying any fault cases

import re

print("Current fault cases:", len(fault_df))

for _, row in high_scoring.iterrows():
    text = str(row["candidate_zh"])

    # Identify useful characteristics
    has_number = bool(re.search(r"\d+", text))
    has_negation = any(
        word in text
        for word in ["不", "没", "无", "未"]
    )
    has_multiple_clauses = any(
        mark in text
        for mark in ["，", ",", "；", ";"]
    )

    print(
        f"{row['eval_id']} | "
        f"Number: {has_number} | "
        f"Negation: {has_negation} | "
        f"Multiple clauses: {has_multiple_clauses}"
    )
    print("English:", row["source_en"])
    print("Mandarin:", text)
    print("-" * 60)


Current fault cases: 6
E001 | Number: False | Negation: False | Multiple clauses: True
English: Democratic members of the House Intelligence Committee released their own findings on Thursday that agreed with some of the conclusions of the Republican task force.
Mandarin: 众议院情报委员会的民主成员在星期四公布了他们自己的调查结果,他们同意共和党工作队的一些结论。
------------------------------------------------------------
E003 | Number: True | Negation: False | Multiple clauses: False
English: Gee and Schwartzman are scheduled to discuss and share images of the 1926 Bertram Goodhue design as well as the 1993 restoration and addition by Hardy Holzman Pfeiffer Associates.
Mandarin: Gee和Schwartzman计划讨论和分享Bertram Goodhue 1926年设计以及Hardy Holzman Pfeiffer Associates1993年修复和补充的图像。
------------------------------------------------------------
E004 | Number: False | Negation: False | Multiple clauses: True
English: Over the past decade, the State Council has issued a total of five notices for the elimination of excessive and backward product

In [6]:
# Continue from our existing six fault cases.
# Do not recreate the original cases.

# Negation: remove the original negative meaning
add_fault(
    "E015",
    "negation",
    "那个白痴知道他们还在直播",
    "Removed 不 from 不知道, reversing the meaning."
)

add_fault(
    "E038",
    "negation",
    "去年,官员们说,胡克尔先生的小组的结论是,尽管公开声明相反,但针对伊斯兰国家控制的炼油厂的空袭大大削弱了恐怖主义集团的财政,因为它建造了临时炼油厂,在黑市上出售石油。",
    "Removed 没有 from 没有大大削弱, reversing the finding."
)

# Number swaps: change one factual value
add_fault(
    "E021",
    "entity_number",
    "截至2016年6月,全世界共进行了114次月球勘探,包括53次火星探测任务。",
    "Changed 43 to 53."
)

add_fault(
    "E046",
    "entity_number",
    "这份20页的报告详述了2014年和2015年在中央司令部描述和分析美国训练伊拉克部队的努力方面长期存在的问题。",
    "Changed 10 pages to 20 pages."
)

# Dropped clauses: remove a meaningful statement
add_fault(
    "E020",
    "dropped_clause",
    "Yuyue和Jiuzhotong之间的合作旨在满足市场的需要,帮助医院建立先进的供应链管理系统。",
    "Removed the clause about seeking further cooperation."
)

add_fault(
    "E035",
    "dropped_clause",
    "这些公司的存在扭曲了市场信号,降低了资源使用效率要素,增加了社会管理的成本。",
    "Removed the concluding clause about hindering fair competition."
)

# Update the existing DataFrame
fault_df = pd.DataFrame(fault_cases)

print("Total applied fault cases:", len(fault_df))

display(
    fault_df.groupby("fault_type").size()
)


Total applied fault cases: 12


,0
fault_type,
dropped_clause,3
entity_number,3
negation,3
punctuation_control,1
truncation,1
untranslated,1


In [7]:

# Add 2 truncation cases
add_fault(
    "E021",
    "truncation",
    "截至2016年6月,全世界共进行了114次月球勘探,",
    "Truncated the translation before the Mars missions."
)

add_fault(
    "E050",
    "truncation",
    "综合付款机构应在通知之日起",
    "Truncated the translation before the six-month deadline."
)

# Add 2 untranslated cases
for eval_id in ["E018", "E032"]:
    row = high_scoring.loc[
        high_scoring["eval_id"] == eval_id
    ].iloc[0]

    add_fault(
        eval_id,
        "untranslated",
        row["source_en"],
        "Returned the original English source instead of Mandarin."
    )

# Add 2 punctuation-only controls
add_fault(
    "E011",
    "punctuation_control",
    "对安卡拉来说,要求休战的好处是显而易见的!",
    "Changed final punctuation only."
)

add_fault(
    "E004",
    "punctuation_control",
    "过去十年来,国务院共发出五份消除过度和落后生产能力的通知!",
    "Changed final punctuation only."
)

# Update our existing DataFrame
fault_df = pd.DataFrame(fault_cases)

print("Total applied cases:", len(fault_df))

display(
    fault_df.groupby("fault_type").size()
)

assert len(fault_df) == 18


Total applied cases: 18


,0
fault_type,
dropped_clause,3
entity_number,3
negation,3
punctuation_control,3
truncation,3
untranslated,3


In [8]:
JUDGE_PROMPT = """
You are evaluating the quality of an English-to-Mandarin Chinese translation.

English source:
{source}

Reference Mandarin translation:
{reference}

Candidate Mandarin translation:
{candidate}

Evaluate the candidate translation using the following criteria:

Accuracy:
Does the candidate correctly preserve the meaning of the English source?

Completeness:
Does the candidate preserve all important information without omissions
or unsupported additions?

Fluency:
Is the Mandarin grammatically correct, natural, and readable?

Terminology:
Are words, expressions, names, and domain-specific terms translated
appropriately?

Based on these criteria, give one overall score from 1 to 5 using the
following scoring anchors:

5 = Excellent
The meaning is fully preserved. There are no meaningful mistranslations,
omissions, or unsupported additions.

4 = Good
The meaning is substantially preserved. Minor issues may exist but do not
materially change the source meaning.

3 = Acceptable
The main meaning is preserved, but noticeable errors reduce translation quality.

2 = Poor
Important information is mistranslated, omitted, or incorrectly added.

1 = Very poor
The translation substantially fails to convey the source meaning.

Return your answer exactly in this format:

Score: <1-5>
Explanation: <brief explanation>
""".strip()

print("P2 Anchored prompt loaded successfully!")

P2 Anchored prompt loaded successfully!


In [9]:

import os, re, json
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "Enable T4 GPU first."

MODEL_NAME = "Qwen/Qwen3-4B"
RESULT_FILE = "fault_detection_raw_results.csv"

# Reuse the exact P2 prompt from Task 2.
# Paste that notebook's JUDGE_PROMPT definition here if
# this is a separate Colab runtime.
assert "JUDGE_PROMPT" in globals(), (
    "Copy the exact JUDGE_PROMPT from Task 2 first."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

# Score both clean and faulty translations.
# Reuse existing fault_df.
applied = fault_df[fault_df["status"] == "APPLIED"].copy()
assert len(applied) == 18

jobs = []
for i, row in applied.reset_index(drop=True).iterrows():
    for condition, translation in [
        ("clean", row["clean_translation"]),
        ("faulty", row["faulty_translation"])
    ]:
        for run in range(1, 4):
            jobs.append({
                "case_id": i,
                "eval_id": row["eval_id"],
                "fault_type": row["fault_type"],
                "condition": condition,
                "run": run,
                "source_en": row["source_en"],
                "reference_zh": row["reference_zh"],
                "translation": translation
            })

def score_translation(job):
    prompt = JUDGE_PROMPT.format(
        source=job["source_en"],
        reference=job["reference_zh"],
        candidate=job["translation"]
    )

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            max_new_tokens=128,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    match = re.search(
        r"\bScore\s*:\s*([1-5])\b",
        response,
        flags=re.IGNORECASE
    )
    score = int(match.group(1)) if match else np.nan
    return score, response

# Save progressively so interrupted runs can resume.
results = (
    pd.read_csv(RESULT_FILE).to_dict("records")
    if os.path.exists(RESULT_FILE) else []
)

done = {
    (int(x["case_id"]), x["condition"], int(x["run"]))
    for x in results
}

for job in tqdm(jobs):
    key = (job["case_id"], job["condition"], job["run"])
    if key in done:
        continue

    score, response = score_translation(job)

    results.append({
        "case_id": job["case_id"],
        "eval_id": job["eval_id"],
        "fault_type": job["fault_type"],
        "condition": job["condition"],
        "run": job["run"],
        "score": score,
        "raw_response": response
    })

    done.add(key)
    if len(results) % 10 == 0:
        pd.DataFrame(results).to_csv(RESULT_FILE, index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(RESULT_FILE, index=False)

print("Total evaluations:", len(results_df))
print("Valid scores:", results_df["score"].notna().sum())
print("Invalid scores:", results_df["score"].isna().sum())


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  0%|          | 0/108 [00:00<?, ?it/s]

Total evaluations: 108
Valid scores: 108
Invalid scores: 0


In [10]:
assert len(results_df) == 108
assert results_df["score"].notna().all()

means = (
    results_df.groupby(
        ["case_id", "eval_id", "fault_type", "condition"]
    )["score"]
    .mean()
    .unstack("condition")
    .reset_index()
)

means["drop_5"] = means["clean"] - means["faulty"]
means["drop_10_equivalent"] = means["drop_5"] * 2.5

thresholds = {
    "untranslated": 7,
    "negation": 5,
    "entity_number": 4,
    "dropped_clause": 3,
    "truncation": 3
}

means["expected_drop_10"] = means["fault_type"].map(thresholds)

# Control criterion: absolute change <= 0.5/10
means["passed"] = means.apply(
    lambda x:
        abs(x["drop_10_equivalent"]) <= 0.5
        if x["fault_type"] == "punctuation_control"
        else x["drop_10_equivalent"] >= x["expected_drop_10"],
    axis=1
)

summary = means.groupby("fault_type").agg(
    cases=("case_id", "count"),
    mean_drop_10=("drop_10_equivalent", "mean"),
    passed=("passed", "sum")
).reset_index()

summary["pass_rate"] = (
    summary["passed"] / summary["cases"] * 100
)

# Save both detailed and category results
means.to_csv("fault_detection_detailed.csv", index=False)
summary.to_csv("fault_detection_summary.csv", index=False)

print("FAULT DETECTION RESULTS")
display(summary.round(2))

print("\nINDIVIDUAL CASE RESULTS")
display(means.round(2))


FAULT DETECTION RESULTS


,fault_type,cases,mean_drop_10,passed,pass_rate
0,dropped_clause,3,0.00,0,0.0
1,entity_number,3,1.67,0,0.0
2,negation,3,2.50,0,0.0
3,punctuation_control,3,0.00,3,100.0
4,truncation,3,2.50,0,0.0
5,untranslated,3,0.83,0,0.0



INDIVIDUAL CASE RESULTS


condition,case_id,eval_id,fault_type,clean,faulty,drop_5,drop_10_equivalent,expected_drop_10,passed
0,0,E008,negation,4.0,3.0,1.0,2.5,5.0,False
1,1,E010,entity_number,4.0,3.0,1.0,2.5,4.0,False
2,2,E001,dropped_clause,4.0,4.0,0.0,0.0,3.0,False
3,3,E004,truncation,4.0,3.0,1.0,2.5,3.0,False
4,4,E011,untranslated,4.0,4.0,0.0,0.0,7.0,False
5,5,E006,punctuation_control,4.0,4.0,0.0,0.0,NaN,True
6,6,E015,negation,4.0,3.0,1.0,2.5,5.0,False
7,7,E038,negation,4.0,3.0,1.0,2.5,5.0,False
8,8,E021,entity_number,4.0,3.0,1.0,2.5,4.0,False
9,9,E046,entity_number,4.0,4.0,0.0,0.0,4.0,False
